# Creating the model with CO2 limit

in this notebook, the model created in model.ipynb will be imported and global_constraints of CO2 limit will be applied.

## Importing packages

In [11]:
import linopy
import pypsa

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import cartopy
import cartopy.crs as ccrs

import networkx as nx

import atlite
from atlite.gis import ExclusionContainer, shape_availability
from rasterio.plot import show
from rasterio.crs import CRS
import rasterio as rio

from pathlib import Path
import xarray as xr

## Importing the base model

In [12]:
n_no_co2 = pypsa.Network("pypsa_model_n.nc") # the model without CO2 limits is imported

INFO:pypsa.network.io:New version 1.0.7 available! (Current: 1.0.5)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, links, loads, storage_units, sub_networks


In [13]:
n_no_co2.global_constraints

attribute,type,investment_period,bus,carrier_attribute,sense,constant,mu
name,,,,,,,


## Adding co2 limit constraint

In [14]:
n_no_co2.add(
    "GlobalConstraint",
    "CO2Limit",
    carrier_attribute="co2_emissions",
    sense="<=",
    constant=0, # the CO2 limit is 0 --> 100% renewable system
)

In [15]:
n_no_co2.global_constraints

,type,investment_period,bus,carrier_attribute,sense,constant,mu
name,,,,,,,
CO2Limit,primary_energy,NaN,,co2_emissions,<=,0.0,0.0


## Solving the model

In [16]:
# solve the model with co2 limit
n_no_co2.optimize(
    #snapshots=n_no_co2.snapshots[:168],  # 1 week
    solver_name="gurobi",
    method=2,
    crossover=0,
    BarConvTol=1.0e-05,
    AggFill=0,
    PreDual=0,
    GURO_PAR_BARDENSETHRESH=200,
    log_to_console=False
)

INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.model:Solver options:
 - method: 2
 - crossover: 0
 - BarConvTol: 1e-05
 - AggFill: 0
 - PreDual: 0
 - GURO_PAR_BARDENSETHRESH: 200
 - log_to_console: False
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 8/8 [00:00<00:00, 50.82it/s]
INFO:linopy.io: Writing time: 3.93s


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2750042


INFO:gurobipy:Set parameter LicenseID to value 2750042


...

Set parameter GURO_PAR_BARDENSETHRESH to value 200


INFO:gurobipy:Set parameter GURO_PAR_BARDENSETHRESH to value 200


Set parameter LogToConsole to value 0


INFO:gurobipy:Set parameter LogToConsole to value 0
INFO:gurobipy:Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 25.2.0 25C56)
INFO:gurobipy:
INFO:gurobipy:CPU model: Apple M4 Pro
INFO:gurobipy:Thread count: 14 physical cores, 14 logical processors, using up to 14 threads
INFO:gurobipy:
INFO:gurobipy:Non-default parameters:
INFO:gurobipy:Method  2
INFO:gurobipy:BarConvTol  1e-05
INFO:gurobipy:Crossover  0
INFO:gurobipy:AggFill  0
INFO:gurobipy:PreDual  0
INFO:gurobipy:LogToConsole  0
INFO:gurobipy:GURO_PAR_BARDENSETHRESH  200
INFO:gurobipy:
INFO:gurobipy:Optimize a model with 5019879 rows, 2193249 columns and 10762913 nonzeros (Min)
INFO:gurobipy:Model fingerprint: 0x32dd7df5
INFO:gurobipy:Model has 318609 linear objective coefficients
INFO:gurobipy:Coefficient statistics:
INFO:gurobipy:  Matrix range     [2e-10, 7e+02]
INFO:gurobipy:  Objective range  [1e-02, 2e+05]
INFO:gurobipy:  Bounds range     [0e+00, 0e+00]
INFO:gurobipy:  RHS range        [2e+00, 4e+05]
I

('ok', 'optimal')

In [17]:
n_no_co2.objective / 1e6 # in Mio. €

29032.13391567267

## Exporting the model into a .nc file

In [18]:
n_no_co2.export_to_netcdf("pypsa_model_n_no_co2.nc")

INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'pypsa_model_n_no_co2.nc contains: carriers, buses, links, sub_networks, storage_units, global_constraints, loads, generators


<xarray.Dataset> Size: 27MB
Dimensions:                            (snapshots: 2920, carriers_i: 10,
                                        buses_i: 31, buses_t_p_i: 31,
                                        buses_t_marginal_price_i: 31,
                                        links_i: 74, links_t_p0_i: 74,
                                        links_t_p1_i: 74, sub_networks_i: 31,
                                        storage_units_i: 186,
                                        ...
                                        storage_units_t_state_of_charge_i: 186,
                                        global_constraints_i: 1, loads_i: 31,
                                        loads_t_p_set_i: 31, loads_t_p_i: 31,
                                        generators_i: 119,
                                        generators_t_p_max_pu_i: 69,
                                        generators_t_p_i: 80)
Coordinates: (12/21)
  * snapshots                          (snapshots) int64 23kB 0 1 ... 2918 2919
  * carriers_i                         (carriers_i) object 80B 'CCGT' ... 'el...
  * buses_i                            (buses_i) object 248B 'Alborz' ... 'Za...
  * buses_t_p_i                        (buses_t_p_i) object 248B 'Alborz' ......
  * buses_t_marginal_price_i           (buses_t_marginal_price_i) object 248B ...
  * links_i                            (links_i) object 592B 'line_Alborz__Ma...
    ...                                 ...
  * loads_i                            (loads_i) object 248B 'load_Alborz' .....
  * loads_t_p_set_i                    (loads_t_p_set_i) object 248B 'load_Al...
  * loads_t_p_i                        (loads_t_p_i) object 248B 'load_Alborz...
  * generators_i                       (generators_i) object 952B 'exist_CCGT...
  * generators_t_p_max_pu_i            (generators_t_p_max_pu_i) object 552B ...
  * generators_t_p_i                   (generators_t_p_i) object 640B 'exist_...
Data variables: (12/56)
    snapshots_snapshot                 (snapshots) datetime64[ns] 23kB 2025-0...
    snapshots_objective                (snapshots) float64 23kB 1.0 1.0 ... 1.0
    snapshots_stores                   (snapshots) float64 23kB 1.0 1.0 ... 1.0
    snapshots_generators               (snapshots) float64 23kB 1.0 1.0 ... 1.0
    carriers_co2_emissions             (carriers_i) float64 80B 0.198 ... 0.0
    carriers_color                     (carriers_i) object 80B 'red' ... 'black'
    ...                                 ...
    generators_marginal_cost           (generators_i) float64 952B 47.65 ... ...
    generators_capital_cost            (generators_i) float64 952B 1.079e+05 ...
    generators_efficiency              (generators_i) float64 952B 0.57 ... 1.0
    generators_p_nom_opt               (generators_i) float64 952B 997.0 ... ...
    generators_t_p_max_pu              (snapshots, generators_t_p_max_pu_i) float64 2MB ...
    generators_t_p                     (snapshots, generators_t_p_i) float64 2MB ...
Attributes:
    network__linearized_uc:       0
    network__multi_invest:        0
    network__objective:           29032133915.672672
    network__objective_constant:  0.0
    network_name:                 Unnamed Network
    network_pypsa_version:        1.0.5
    network_srid:                 4326
    crs:                          {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"Wor...
    meta:                         {}